<a href="https://colab.research.google.com/github/ayoitshasya/respiratory-disorder-detection/blob/main/pulmoscan_multitask_hf_lung_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/pulmoscan semester 8/colab_training_package.zip" -d /content/project
%cd /content/project
!ls -la

/content/project
total 20
drwxr-xr-x 5 root root 4096 Aug 31 20:12 .
drwxr-xr-x 1 root root 4096 Aug 31 20:12 ..
drwxr-xr-x 4 root root 4096 Aug 31 20:12 data
drwxr-xr-x 2 root root 4096 Aug 31 20:12 scripts
drwxr-xr-x 2 root root 4096 Aug 31 20:12 src


In [ ]:
!pip install -q librosa scikit-learn tqdm

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", gpus)
assert len(gpus) > 0, "No GPU detected — check Runtime > Change runtime type > T4 GPU"

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!grep -n "manifest" scripts/train_multitask.py

22:MANIFEST_PATH = 'data/processed/manifest_merged.csv'
87:manifest = pd.read_csv(MANIFEST_PATH)
88:train_df  = manifest[manifest['split'] == 'train']
89:val_df    = manifest[manifest['split'] == 'val']
236:test_df = manifest[manifest['split'] == 'test']


In [ ]:
!python scripts/train_multitask.py

Train: 27810 | Val: 9106
Train distribution:
sound_label
0    12216
1     9977
2     5197
3      420
Name: count, dtype: int64
Loading val...
Traceback (most recent call last):
  File "/content/project/scripts/train_multitask.py", line 107, in <module>
    X_val, y_sv, y_dv = load_split(val_df)
                        ~~~~~~~~~~^^^^^^^^
  File "/content/project/scripts/train_multitask.py", line 98, in load_split
    feat = np.load(row['features_path'])
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 459, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
                              ~~~~^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Hasya Abburi\\Desktop\\leo\\kjsce\\academics\\MAJOR PROJECT - RESPIRATORY DISORDER DETECTION\\respiratory disorder detection - pulmoscan\\data\\processed\\val\\110_1b1_Pr_sc_Meditron_cycle0_crackle.npy'


In [ ]:
import pandas as pd
import re

manifest = pd.read_csv('data/processed/manifest_merged.csv')

def fix_path(p):
    # Extract everything from 'data\' or 'data/' onwards, convert to Colab-relative path
    if pd.isna(p):
        return p
    p = str(p)
    # find the 'data' segment regardless of Windows/Unix separators
    match = re.search(r'[Dd]ata[\\/].*', p)
    if match:
        relative = match.group(0).replace('\\', '/')
        return '/content/project/' + relative
    return p

before_sample = manifest['features_path'].iloc[0]
manifest['features_path'] = manifest['features_path'].apply(fix_path)
manifest['audio_path'] = manifest['audio_path'].apply(fix_path)
after_sample = manifest['features_path'].iloc[0]

print("Before:", before_sample)
print("After: ", after_sample)

# Verify a sample of paths actually exist before saving
import os
missing = manifest['features_path'].apply(lambda p: not os.path.exists(p)).sum()
print(f"\nMissing files after fix: {missing} / {len(manifest)}")

if missing == 0:
    manifest.to_csv('data/processed/manifest_merged.csv', index=False)
    print("Saved corrected manifest.")
else:
    print("DO NOT proceed — some paths still don't resolve. Investigate before saving.")

Before: C:\Users\Hasya Abburi\Desktop\leo\kjsce\academics\MAJOR PROJECT - RESPIRATORY DISORDER DETECTION\respiratory disorder detection - pulmoscan\data\processed\train\102_1b1_Ar_sc_Meditron_cycle0_normal.npy
After:  /content/project/data/processed/train/102_1b1_Ar_sc_Meditron_cycle0_normal.npy

Missing files after fix: 0 / 46852
Saved corrected manifest.


In [ ]:
!python scripts/train_multitask.py

Train: 27810 | Val: 9106
Train distribution:
sound_label
0    12216
1     9977
2     5197
3      420
Name: count, dtype: int64
Loading val...
Val: (9106, 128, 63, 1)
Loading train...
Train: (27810, 128, 63, 1)
Diagnosis-labeled rows: train 2891/27810, val 799/9106
2026-08-31 17:10:38.839761: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1788196238.841230    7180 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Class weights: [0.5691306483300589, 0.6968527613511075, 1.3377910332884357, 16.553571428571427]

Starting training...
2026-08-31 17:10:41.762520: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 897039360 exceeds 10% of free system memory.
2026-08-31 17:10:42.808746: 

In [ ]:
!grep -n "model_best(X_val\|model_best(X_test" scripts/train_multitask.py

210:val_preds  = model_best(X_val, training=False)
241:test_preds  = model_best(X_test, training=False)


In [ ]:
with open('scripts/train_multitask.py', 'r') as f:
    content = f.read()

replacements = [
    (
        "val_preds  = model_best(X_val, training=False)\n"
        "    y_pred     = np.argmax(val_preds[0].numpy(), axis=1)",
        "val_preds  = model_best.predict(X_val, batch_size=128, verbose=0)\n"
        "    y_pred     = np.argmax(val_preds[0], axis=1)"
    ),
    (
        "y_pred_d   = np.argmax(val_preds[1].numpy(), axis=1)",
        "y_pred_d   = np.argmax(val_preds[1], axis=1)"
    ),
    (
        "test_preds  = model_best(X_test, training=False)\n"
        "    y_pred_test = np.argmax(test_preds[0].numpy(), axis=1)",
        "test_preds  = model_best.predict(X_test, batch_size=128, verbose=0)\n"
        "    y_pred_test = np.argmax(test_preds[0], axis=1)"
    ),
]

applied = 0
for old, new in replacements:
    if old in content:
        content = content.replace(old, new)
        applied += 1
    else:
        print(f"WARNING: pattern not found, skipped:\n{old[:80]}...\n")

print(f"Applied {applied}/3 replacements")

with open('scripts/train_multitask.py', 'w') as f:
    f.write(content)

# Verify the fix landed
!grep -n "model_best.predict\|model_best(X_val\|model_best(X_test" scripts/train_multitask.py

val_preds  = model_best(X_val, training=False)
    y_pred     = np.argmax(val_pr...

test_preds  = model_best(X_test, training=False)
    y_pred_test = np.argmax(tes...

Applied 1/3 replacements
210:val_preds  = model_best(X_val, training=False)
241:test_preds  = model_best(X_test, training=False)


In [ ]:
!sed -n '208,215p' scripts/train_multitask.py

print(f'\nBest val_icbhi: {max(history.history["val_icbhi"]):.4f}')
model_best = tf.keras.models.load_model(f'{CKPT_DIR}/{CKPT_NAME}', compile=False)
val_preds  = model_best(X_val, training=False)
y_pred     = np.argmax(val_preds[0].numpy(), axis=1)
print('\nSound Classification Report:')
print(classification_report(y_sv, y_pred, target_names=SOUND_NAMES))
y_pred_d   = np.argmax(val_preds[1], axis=1)
print('Diagnosis Classification Report:')


In [ ]:
!sed -n '239,246p' scripts/train_multitask.py

print(f'Test: {X_test.shape}  ({n_icbhi_test} icbhi, {n_hf_test} hf_lung)')

test_preds  = model_best(X_test, training=False)
y_pred_test = np.argmax(test_preds[0].numpy(), axis=1)


def report_subset(name, mask):
    yt, yp = y_s_test[mask], y_pred_test[mask]


In [ ]:
with open('scripts/train_multitask.py', 'r') as f:
    content = f.read()

replacements = [
    (
        "val_preds  = model_best(X_val, training=False)\n"
        "y_pred     = np.argmax(val_preds[0].numpy(), axis=1)",
        "val_preds  = model_best.predict(X_val, batch_size=128, verbose=0)\n"
        "y_pred     = np.argmax(val_preds[0], axis=1)"
    ),
    (
        "y_pred_d   = np.argmax(val_preds[1].numpy(), axis=1)",
        "y_pred_d   = np.argmax(val_preds[1], axis=1)"
    ),
    (
        "test_preds  = model_best(X_test, training=False)\n"
        "y_pred_test = np.argmax(test_preds[0].numpy(), axis=1)",
        "test_preds  = model_best.predict(X_test, batch_size=128, verbose=0)\n"
        "y_pred_test = np.argmax(test_preds[0], axis=1)"
    ),
]

applied = 0
for old, new in replacements:
    if old in content:
        content = content.replace(old, new)
        applied += 1
    else:
        print(f"WARNING: not found:\n{repr(old)}\n")

print(f"Applied {applied}/3 replacements")

with open('scripts/train_multitask.py', 'w') as f:
    f.write(content)

'y_pred_d   = np.argmax(val_preds[1].numpy(), axis=1)'

Applied 2/3 replacements


In [ ]:
!grep -n "y_pred_d" scripts/train_multitask.py

214:y_pred_d   = np.argmax(val_preds[1], axis=1)
219:print(classification_report(y_dv[diag_mask_val], y_pred_d[diag_mask_val],
327:cm_diag = confusion_matrix(y_dv[diag_mask_val], y_pred_d[diag_mask_val], labels=list(range(NUM_DIAGNOSIS)))


In [ ]:
!sed -n '208,220p' scripts/train_multitask.py

print(f'\nBest val_icbhi: {max(history.history["val_icbhi"]):.4f}')
model_best = tf.keras.models.load_model(f'{CKPT_DIR}/{CKPT_NAME}', compile=False)
val_preds  = model_best.predict(X_val, batch_size=128, verbose=0)
y_pred     = np.argmax(val_preds[0], axis=1)
print('\nSound Classification Report:')
print(classification_report(y_sv, y_pred, target_names=SOUND_NAMES))
y_pred_d   = np.argmax(val_preds[1], axis=1)
print('Diagnosis Classification Report:')
# Exclude rows with no real diagnosis label (e.g. HF_Lung val rows) — their
# diagnosis_label is the DIAG_MISSING sentinel, not a class to score against.
# No-op when diag_mask_val is all-True (ICBHI-only data).
print(classification_report(y_dv[diag_mask_val], y_pred_d[diag_mask_val],
                             labels=list(range(NUM_DIAGNOSIS)), target_names=DIAG_NAMES))


In [ ]:
!sed -n '239,247p' scripts/train_multitask.py

print(f'Test: {X_test.shape}  ({n_icbhi_test} icbhi, {n_hf_test} hf_lung)')

test_preds  = model_best.predict(X_test, batch_size=128, verbose=0)
y_pred_test = np.argmax(test_preds[0], axis=1)


def report_subset(name, mask):
    yt, yp = y_s_test[mask], y_pred_test[mask]
    score = icbhi_score(yt, yp)


In [ ]:
!grep -n "^model_best = tf.keras.models.load_model\|history = model.fit\|^model =" scripts/train_multitask.py

169:model = build_model()
199:history = model.fit(
209:model_best = tf.keras.models.load_model(f'{CKPT_DIR}/{CKPT_NAME}', compile=False)


In [ ]:
%%writefile scripts/eval_only.py
import sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

# --- Pull in shared constants/functions from the training script without re-running training ---
# Copy these exactly from train_multitask.py's top section
MANIFEST_PATH = 'data/processed/manifest_merged.csv'
CKPT_DIR = 'data/checkpoints'
CKPT_NAME = 'multitask_combined_best.keras'
NUM_SOUND = 4
NUM_DIAGNOSIS = 7
SOUND_NAMES = ['Normal', 'Crackle', 'Wheeze', 'Both']
DIAG_MISSING = -1

print("Loading manifest and val/test splits...")
manifest = pd.read_csv(MANIFEST_PATH)
val_df  = manifest[manifest['split'] == 'val']
test_df = manifest[manifest['split'] == 'test']

def load_split(df):
    X, y_s, y_d = [], [], []
    for _, row in df.iterrows():
        feat = np.load(row['features_path'])
        X.append(feat)
        y_s.append(row['sound_label'])
        y_d.append(row['diagnosis_label'])
    return np.array(X)[..., np.newaxis], np.array(y_s), np.array(y_d)

print("Loading val...")
X_val, y_sv, y_dv = load_split(val_df)
print("Val:", X_val.shape)

print("Loading test...")
X_test, y_s_test, y_d_test = load_split(test_df)
print("Test:", X_test.shape)

diag_mask_val  = (y_dv != DIAG_MISSING)
diag_mask_test = (y_d_test != DIAG_MISSING)

print(f"\nLoading checkpoint: {CKPT_DIR}/{CKPT_NAME}")
model_best = tf.keras.models.load_model(f'{CKPT_DIR}/{CKPT_NAME}', compile=False)

print("\nRunning val predictions...")
val_preds = model_best.predict(X_val, batch_size=128, verbose=0)
y_pred = np.argmax(val_preds[0], axis=1)
print('\nSound Classification Report (val):')
print(classification_report(y_sv, y_pred, target_names=SOUND_NAMES))

print("\nRunning test predictions...")
test_preds = model_best.predict(X_test, batch_size=128, verbose=0)
y_pred_test = np.argmax(test_preds[0], axis=1)
print('\nSound Classification Report (test, full combined):')
print(classification_report(y_s_test, y_pred_test, target_names=SOUND_NAMES))

def icbhi_score(yt, yp):
    normal_mask = (yt == 0)
    abnormal_mask = (yt != 0)
    sp = (yp[normal_mask] == 0).mean() if normal_mask.sum() > 0 else 0
    se = (yp[abnormal_mask] != 0).mean() if abnormal_mask.sum() > 0 else 0
    return (se + sp) / 2

def report_subset(name, mask):
    yt, yp = y_s_test[mask], y_pred_test[mask]
    score = icbhi_score(yt, yp)
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_SOUND)))
    print(f"\n=== {name} (n={mask.sum()}) ===")
    print(f"ICBHI score: {score:.4f}")
    print("Confusion matrix:\n", cm)
    return score

print("\n\n========== TEST SET BREAKDOWN ==========")
full_score  = report_subset("Full combined test set", np.ones(len(y_s_test), dtype=bool))
icbhi_mask  = (test_df['source_dataset'] == 'icbhi').values
hflung_mask = (test_df['source_dataset'] == 'hf_lung').values
icbhi_score_only  = report_subset("ICBHI-only subset",   icbhi_mask)
hflung_score_only = report_subset("HF_Lung-only subset", hflung_mask)

print("\n\n========== COMPARISON TO SEM-6 BASELINE ==========")
print(f"Sem-6 baseline (ICBHI-only, val split):     62.26%")
print(f"This run — full combined test:              {full_score*100:.2f}%")
print(f"This run — ICBHI-only test subset:           {icbhi_score_only*100:.2f}%")
print(f"This run — HF_Lung-only test subset:         {hflung_score_only*100:.2f}%")

# Normal -> Crackle confusion count, ICBHI-only
yt_icbhi = y_s_test[icbhi_mask]
yp_icbhi = y_pred_test[icbhi_mask]
normal_to_crackle = ((yt_icbhi == 0) & (yp_icbhi == 1)).sum()
total_normal_icbhi = (yt_icbhi == 0).sum()
print(f"\nNormal->Crackle confusion (ICBHI-only test): {normal_to_crackle}/{total_normal_icbhi}")
print(f"(Sem-6 baseline was 171/506 on val split)")

Writing scripts/eval_only.py


In [ ]:
!python scripts/eval_only.py

Loading manifest and val/test splits...
Loading val...
Traceback (most recent call last):
  File "/content/project/scripts/eval_only.py", line 33, in <module>
    X_val, y_sv, y_dv = load_split(val_df)
                        ~~~~~~~~~~^^^^^^^^
  File "/content/project/scripts/eval_only.py", line 30, in load_split
    return np.array(X)[..., np.newaxis], np.array(y_s), np.array(y_d)
           ~~~~~~~~^^^
ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (9106, 128) + inhomogeneous part.


In [ ]:
import numpy as np
import pandas as pd

manifest = pd.read_csv('data/processed/manifest_merged.csv')
val_df = manifest[manifest['split'] == 'val']

shapes = {}
bad_rows = []
for idx, row in val_df.iterrows():
    feat = np.load(row['features_path'])
    shapes[feat.shape] = shapes.get(feat.shape, 0) + 1
    if feat.shape != (128, 63):
        bad_rows.append((row['features_path'], feat.shape, row.get('source_dataset', '?')))

print("Shape distribution across val set:")
for shape, count in sorted(shapes.items(), key=lambda x: -x[1]):
    print(f"  {shape}: {count} files")

print(f"\nTotal mismatched files: {len(bad_rows)}")
if bad_rows:
    print("\nFirst 10 examples:")
    for path, shape, source in bad_rows[:10]:
        print(f"  {source}: {shape} — {path}")

Shape distribution across val set:
  (128, 63): 8307 files
  (128, 126): 148 files
  (128, 259): 15 files
  (128, 181): 12 files
  (128, 152): 12 files
  (128, 204): 12 files
  (128, 267): 12 files
  (128, 134): 11 files
  (128, 247): 10 files
  (128, 164): 10 files
  (128, 140): 10 files
  (128, 226): 9 files
  (128, 248): 9 files
  (128, 182): 9 files
  (128, 201): 9 files
  (128, 239): 9 files
  (128, 209): 9 files
  (128, 141): 9 files
  (128, 179): 9 files
  (128, 221): 8 files
  (128, 257): 8 files
  (128, 275): 8 files
  (128, 233): 8 files
  (128, 136): 8 files
  (128, 245): 7 files
  (128, 230): 7 files
  (128, 137): 7 files
  (128, 157): 7 files
  (128, 242): 7 files
  (128, 238): 7 files
  (128, 223): 7 files
  (128, 225): 7 files
  (128, 253): 7 files
  (128, 235): 7 files
  (128, 139): 6 files
  (128, 144): 6 files
  (128, 243): 6 files
  (128, 272): 5 files
  (128, 220): 5 files
  (128, 258): 5 files
  (128, 151): 5 files
  (128, 162): 5 files
  (128, 169): 5 files
  (128

In [ ]:
!grep -n "def load_split" -A 30 scripts/train_multitask.py

95:def load_split(df):
96-    X, ys, yd = [], [], []
97-    for _, row in df.iterrows():
98-        feat = np.load(row['features_path'])
99-        feat = pad_or_truncate(feat)
100-        if feat.ndim == 2:
101-            feat = feat[..., np.newaxis]
102-        X.append(feat); ys.append(row['sound_label']); yd.append(row['diagnosis_label'])
103-    return np.array(X, np.float32), np.array(ys, np.int32), np.array(yd, np.int32)
104-
105-
106-print('Loading val...')
107-X_val, y_sv, y_dv = load_split(val_df)
108-print(f'Val: {X_val.shape}')
109-
110-print('Loading train...')
111-X_tr, y_str, y_dtr = load_split(train_df)
112-print(f'Train: {X_tr.shape}')
113-
114-y_str_oh = tf.keras.utils.to_categorical(y_str, NUM_SOUND)
115-y_sv_oh  = tf.keras.utils.to_categorical(y_sv,  NUM_SOUND)
116-
117-# Diagnosis masking: rows with no real diagnosis label (diagnosis_label ==
118-# DIAG_MISSING, e.g. HF_Lung rows) get an all-zero one-hot target instead of
119-# a real class. With y_true all-zero, 

In [ ]:
!grep -n "def pad_or_truncate" -A 15 scripts/train_multitask.py

37:def pad_or_truncate(feat, t=TARGET_FRAMES):
38-    c = feat.shape[-1]
39-    if c < t:
40-        feat = np.pad(feat, [(0,0), (0, t-c)])
41-    else:
42-        feat = feat[..., :t]
43-    return feat
44-
45-
46-def focal_loss(gamma=2.0, class_weights=None):
47-    def loss_fn(y_true, y_pred):
48-        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
49-        ce  = -y_true * tf.math.log(y_pred)
50-        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
51-        focal = tf.pow(1.0 - p_t, gamma) * ce
52-        if class_weights is not None:


In [ ]:
%%writefile scripts/eval_only.py
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

MANIFEST_PATH = 'data/processed/manifest_merged.csv'
CKPT_DIR = 'data/checkpoints'
CKPT_NAME = 'multitask_combined_best.keras'
NUM_SOUND = 4
TARGET_FRAMES = 63
SOUND_NAMES = ['Normal', 'Crackle', 'Wheeze', 'Both']

def pad_or_truncate(feat, t=TARGET_FRAMES):
    c = feat.shape[-1]
    if c < t:
        feat = np.pad(feat, [(0,0), (0, t-c)])
    else:
        feat = feat[..., :t]
    return feat

def load_split_with_source(df):
    X, ys, src = [], [], []
    for _, row in df.iterrows():
        feat = np.load(row['features_path'])
        feat = pad_or_truncate(feat)
        if feat.ndim == 2:
            feat = feat[..., np.newaxis]
        X.append(feat); ys.append(row['sound_label']); src.append(row['source_dataset'])
    return np.array(X, np.float32), np.array(ys, np.int32), np.array(src)

def icbhi_score(yt, yp):
    normal_mask = (yt == 0)
    abnormal_mask = (yt != 0)
    sp = (yp[normal_mask] == 0).mean() if normal_mask.sum() > 0 else 0
    se = (yp[abnormal_mask] != 0).mean() if abnormal_mask.sum() > 0 else 0
    return (se + sp) / 2

def report_subset(name, mask, y_s_test, y_pred_test):
    yt, yp = y_s_test[mask], y_pred_test[mask]
    score = icbhi_score(yt, yp)
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_SOUND)))
    print(f'\n=== TEST subset: {name} (n={mask.sum()}) ===')
    print(f'  ICBHI score: {score:.4f} ({score*100:.2f}%)')
    print(f'  Normal->Crackle confusions: {cm[0, 1]} / {cm[0].sum()} true Normal')
    print('  Confusion matrix (rows=true, cols=pred), order Normal/Crackle/Wheeze/Both:')
    print(cm)
    print(classification_report(yt, yp, target_names=SOUND_NAMES, zero_division=0))
    return score

print("Loading manifest...")
manifest = pd.read_csv(MANIFEST_PATH)
test_df = manifest[manifest['split'] == 'test']

print(f"\nLoading checkpoint: {CKPT_DIR}/{CKPT_NAME}")
model_best = tf.keras.models.load_model(f'{CKPT_DIR}/{CKPT_NAME}', compile=False)

print('\nLoading test...')
X_test, y_s_test, src_test = load_split_with_source(test_df)
n_icbhi_test, n_hf_test = (src_test == 'icbhi').sum(), (src_test == 'hf_lung').sum()
print(f'Test: {X_test.shape}  ({n_icbhi_test} icbhi, {n_hf_test} hf_lung)')

test_preds  = model_best.predict(X_test, batch_size=128, verbose=0)
y_pred_test = np.argmax(test_preds[0], axis=1)

full_score  = report_subset("Full combined", np.ones(len(y_s_test), dtype=bool), y_s_test, y_pred_test)
icbhi_only  = report_subset("ICBHI-only",   src_test == 'icbhi',   y_s_test, y_pred_test)
hflung_only = report_subset("HF_Lung-only", src_test == 'hf_lung', y_s_test, y_pred_test)

print("\n\n========== COMPARISON TO SEM-6 BASELINE ==========")
print(f"Sem-6 baseline (ICBHI-only, val split): 62.26%")
print(f"This run — full combined test:          {full_score*100:.2f}%")
print(f"This run — ICBHI-only test subset:      {icbhi_only*100:.2f}%")
print(f"This run — HF_Lung-only test subset:    {hflung_only*100:.2f}%")

Overwriting scripts/eval_only.py


In [ ]:
!python scripts/eval_only.py

Loading manifest...

Loading checkpoint: data/checkpoints/multitask_combined_best.keras
2026-08-31 17:36:04.412372: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1788197764.413868   14550 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5

Loading test...
Test: (9936, 128, 63, 1)  (1629 icbhi, 8307 hf_lung)
I0000 00:00:1788197773.530073   14591 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.

=== TEST subset: Full combined (n=9936) ===
  ICBHI score: 0.7164 (71.64%)
  Normal->Crackle confusions: 313 / 4298 true Normal
  Confusion matrix (rows=true, cols=pred), order Normal/Crackle/Wheeze/Both:
[[3063  313  388  534]
 [1236 1363  392  37

In [ ]:
%%writefile scripts/train_multitask.py
"""
Final multitask training — baseline CNN architecture that works,
trained on original 2,891 samples with both sound + diagnosis heads.
Using original data (not augmented) to avoid overfitting collapse.
"""
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

SEED=42; NUM_SOUND=4; NUM_DIAGNOSIS=7; BATCH_SIZE=32
EPOCHS=80; LR=1e-3; LR_MIN=1e-6; TARGET_FRAMES=63; N_MELS=128
DIAG_WEIGHT=0.1; CKPT_DIR='data/checkpoints'; RESULTS_DIR='data/results'
DIAG_MISSING = -1
MANIFEST_PATH = 'data/processed/manifest_merged.csv'
RUN_SUFFIX = 'sourceweighted'
CKPT_NAME = f'multitask_{RUN_SUFFIX}_best.keras'
PREDICT_BATCH_SIZE = 128
BASELINE_ICBHI_SCORE = 0.6226
BASELINE_NORMAL_TO_CRACKLE = (171, 506)
UNWEIGHTED_COMBINED_SCORES = {'full': 0.7164, 'icbhi': 0.5196, 'hf_lung': 0.7608}
ICBHI_SOURCE_WEIGHT = 9.0
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
SOUND_NAMES = ['Normal', 'Crackle', 'Wheeze', 'Both']
DIAG_NAMES  = ['Healthy', 'COPD', 'URTI', 'Bronchiectasis', 'Pneumonia', 'Bronchiolitis', 'Other']


def pad_or_truncate(feat, t=TARGET_FRAMES):
    c = feat.shape[-1]
    if c < t:
        feat = np.pad(feat, [(0,0), (0, t-c)])
    else:
        feat = feat[..., :t]
    return feat


def focal_loss(gamma=2.0, class_weights=None):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce  = -y_true * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        focal = tf.pow(1.0 - p_t, gamma) * ce
        if class_weights is not None:
            weights = tf.reduce_sum(y_true * tf.constant(class_weights, dtype=tf.float32), axis=-1, keepdims=True)
            focal = focal * weights
        return tf.reduce_mean(focal, axis=-1)
    loss_fn.__name__ = 'focal_loss'
    return loss_fn


def icbhi_score(y_true, y_pred, n=NUM_SOUND):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    se, sp = [], []
    for i in range(n):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-tp-fn-fp
        se.append(tp/(tp+fn) if tp+fn>0 else 0.0)
        sp.append(tn/(tn+fp) if tn+fp>0 else 0.0)
    return (np.mean(se)+np.mean(sp))/2.0


class ICBHICallback(tf.keras.callbacks.Callback):
    def __init__(self, X, y):
        super().__init__(); self.X=X; self.y=y
        self.best = 0.0
    def on_epoch_end(self, epoch, logs=None):
        pred  = np.argmax(self.model.predict(self.X, verbose=0)[0], axis=1)
        score = icbhi_score(self.y, pred)
        logs['val_icbhi'] = score
        if score > self.best:
            self.best = score
            self.model.save(f'{CKPT_DIR}/{CKPT_NAME}')
            print(f'  val_icbhi: {score:.4f} *** NEW BEST — saved ***')
        else:
            print(f'  val_icbhi: {score:.4f}')


manifest = pd.read_csv(MANIFEST_PATH)
train_df  = manifest[manifest['split'] == 'train']
val_df    = manifest[manifest['split'] == 'val']
print(f'Train: {len(train_df)} | Val: {len(val_df)}')
print('Train distribution:')
print(train_df['sound_label'].value_counts().sort_index())


def load_split(df):
    X, ys, yd, src = [], [], [], []
    for _, row in df.iterrows():
        feat = np.load(row['features_path'])
        feat = pad_or_truncate(feat)
        if feat.ndim == 2:
            feat = feat[..., np.newaxis]
        X.append(feat); ys.append(row['sound_label']); yd.append(row['diagnosis_label'])
        src.append(row['source_dataset'])
    return (np.array(X, np.float32), np.array(ys, np.int32), np.array(yd, np.int32), np.array(src))


print('Loading val...')
X_val, y_sv, y_dv, src_val = load_split(val_df)
print(f'Val: {X_val.shape}')

print('Loading train...')
X_tr, y_str, y_dtr, src_tr = load_split(train_df)
print(f'Train: {X_tr.shape}')

y_str_oh = tf.keras.utils.to_categorical(y_str, NUM_SOUND)
y_sv_oh  = tf.keras.utils.to_categorical(y_sv,  NUM_SOUND)

diag_mask_train = (y_dtr != DIAG_MISSING)
diag_mask_val   = (y_dv  != DIAG_MISSING)

y_dtr_oh = tf.keras.utils.to_categorical(np.where(diag_mask_train, y_dtr, 0), NUM_DIAGNOSIS)
y_dv_oh  = tf.keras.utils.to_categorical(np.where(diag_mask_val,   y_dv,  0), NUM_DIAGNOSIS)
y_dtr_oh[~diag_mask_train] = 0.0
y_dv_oh[~diag_mask_val]    = 0.0

print(f'Diagnosis-labeled rows: train {diag_mask_train.sum()}/{len(diag_mask_train)}, '
      f'val {diag_mask_val.sum()}/{len(diag_mask_val)}')


def build_model():
    inp = tf.keras.Input(shape=(N_MELS, TARGET_FRAMES, 1))
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    shared = tf.keras.layers.Dense(256, activation='relu')(x)
    shared = tf.keras.layers.Dropout(0.5)(shared)

    s = tf.keras.layers.Dense(128, activation='relu')(shared)
    s = tf.keras.layers.Dropout(0.3)(s)
    sound_out = tf.keras.layers.Dense(NUM_SOUND, activation='softmax', name='sound')(s)

    d = tf.keras.layers.Dense(128, activation='relu')(shared)
    d = tf.keras.layers.Dropout(0.3)(d)
    diag_out = tf.keras.layers.Dense(NUM_DIAGNOSIS, activation='softmax', name='diagnosis')(d)

    return tf.keras.Model(inp, [sound_out, diag_out])


model = build_model()


def lr_schedule(epoch):
    cos = np.cos(np.pi * epoch / EPOCHS)
    return float(LR_MIN + 0.5*(LR - LR_MIN)*(1 + cos))


callbacks = [
    ICBHICallback(X_val, y_sv),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_icbhi', mode='max',
        patience=20, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.LearningRateScheduler(lr_schedule, verbose=0),
]

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y_str), y=y_str)
cw_list = [float(cw[i]) for i in range(NUM_SOUND)]
print(f'Class weights: {cw_list}')

SOURCE_WEIGHT_MAP = {'icbhi': ICBHI_SOURCE_WEIGHT, 'hf_lung': 1.0}
class_weight_per_sample  = np.array([cw_list[label] for label in y_str], dtype=np.float32)
source_weight_per_sample = np.array([SOURCE_WEIGHT_MAP[s] for s in src_tr], dtype=np.float32)
sound_sample_weight = class_weight_per_sample * source_weight_per_sample
print(f'Source weight map: {SOURCE_WEIGHT_MAP}')
print(f'Sound sample_weight: min={sound_sample_weight.min():.3f}, '
      f'max={sound_sample_weight.max():.3f}, mean={sound_sample_weight.mean():.3f}')

diag_sample_weight = np.ones(len(y_str), dtype=np.float32)

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss={'sound': focal_loss(4.0), 'diagnosis': focal_loss(2.0)},
    loss_weights={'sound': 1.0, 'diagnosis': DIAG_WEIGHT},
    metrics={'sound': 'accuracy', 'diagnosis': 'accuracy'}
)

print('\nStarting training...')
history = model.fit(
    X_tr, {'sound': y_str_oh, 'diagnosis': y_dtr_oh},
    sample_weight={'sound': sound_sample_weight, 'diagnosis': diag_sample_weight},
    validation_data=(X_val, {'sound': y_sv_oh, 'diagnosis': y_dv_oh}),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print(f'\nBest val_icbhi: {max(history.history["val_icbhi"]):.4f}')
model_best = tf.keras.models.load_model(f'{CKPT_DIR}/{CKPT_NAME}', compile=False)
val_preds  = model_best.predict(X_val, batch_size=PREDICT_BATCH_SIZE, verbose=0)
y_pred     = np.argmax(val_preds[0], axis=1)
print('\nSound Classification Report:')
print(classification_report(y_sv, y_pred, target_names=SOUND_NAMES))
y_pred_d   = np.argmax(val_preds[1], axis=1)
print('Diagnosis Classification Report:')
print(classification_report(y_dv[diag_mask_val], y_pred_d[diag_mask_val],
                             labels=list(range(NUM_DIAGNOSIS)), target_names=DIAG_NAMES))


def load_split_with_source(df):
    X, ys, src = [], [], []
    for _, row in df.iterrows():
        feat = np.load(row['features_path'])
        feat = pad_or_truncate(feat)
        if feat.ndim == 2:
            feat = feat[..., np.newaxis]
        X.append(feat); ys.append(row['sound_label']); src.append(row['source_dataset'])
    return np.array(X, np.float32), np.array(ys, np.int32), np.array(src)


print('\nLoading test...')
test_df = manifest[manifest['split'] == 'test']
X_test, y_s_test, src_test = load_split_with_source(test_df)
n_icbhi_test, n_hf_test = (src_test == 'icbhi').sum(), (src_test == 'hf_lung').sum()
print(f'Test: {X_test.shape}  ({n_icbhi_test} icbhi, {n_hf_test} hf_lung)')

test_preds  = model_best.predict(X_test, batch_size=PREDICT_BATCH_SIZE, verbose=0)
y_pred_test = np.argmax(test_preds[0], axis=1)


def report_subset(name, mask):
    yt, yp = y_s_test[mask], y_pred_test[mask]
    score = icbhi_score(yt, yp)
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_SOUND)))
    print(f'\n=== TEST subset: {name} (n={mask.sum()}) ===')
    print(f'  ICBHI score: {score:.4f} ({score*100:.2f}%)')
    print(f'  Normal->Crackle confusions: {cm[0, 1]} / {cm[0].sum()} true Normal')
    print('  Confusion matrix (rows=true, cols=pred), order Normal/Crackle/Wheeze/Both:')
    print(cm)
    print(classification_report(yt, yp, target_names=SOUND_NAMES, zero_division=0))
    return score, cm


full_score, full_cm   = report_subset('FULL (icbhi+hf_lung)', np.ones(len(y_s_test), dtype=bool))
icbhi_score_, icbhi_cm = report_subset("source_dataset=='icbhi'", src_test == 'icbhi')
hf_score, hf_cm        = report_subset("source_dataset=='hf_lung'", src_test == 'hf_lung')

print('\n=== Three-way comparison: sem-6 baseline vs. unweighted combined vs. this (source-weighted) run ===')
print('  NOTE: sem-6 baseline (62.26%, 171/506) was measured on the ORIGINAL ICBHI-only')
print('  VAL split (799 rows). The unweighted-combined and this-run numbers are both on')
print('  the (larger, merged) TEST split.')
print(f"  {'subset':22s} {'sem-6 baseline':>15s} {'unweighted combined':>21s} {'this run (source-wt)':>22s}")
print(f"  {'FULL (icbhi+hf_lung)':22s} {'—':>15s} "
      f"{UNWEIGHTED_COMBINED_SCORES['full']*100:20.2f}% {full_score*100:21.2f}%")
print(f"  {'icbhi-only':22s} {BASELINE_ICBHI_SCORE*100:14.2f}% "
      f"{UNWEIGHTED_COMBINED_SCORES['icbhi']*100:20.2f}% {icbhi_score_*100:21.2f}%")
print(f"  {'hf_lung-only':22s} {'—':>15s} "
      f"{UNWEIGHTED_COMBINED_SCORES['hf_lung']*100:20.2f}% {hf_score*100:21.2f}%")
print()
print(f"  icbhi-only delta vs sem-6 baseline:        {(icbhi_score_-BASELINE_ICBHI_SCORE)*100:+.2f} pts")
print(f"  icbhi-only delta vs unweighted combined:   {(icbhi_score_-UNWEIGHTED_COMBINED_SCORES['icbhi'])*100:+.2f} pts")
print(f"  full delta vs unweighted combined:         {(full_score-UNWEIGHTED_COMBINED_SCORES['full'])*100:+.2f} pts")
print(f"  hf_lung-only delta vs unweighted combined: {(hf_score-UNWEIGHTED_COMBINED_SCORES['hf_lung'])*100:+.2f} pts")
print()
print(f'  Baseline Normal->Crackle (val, icbhi-only, sem-6): {BASELINE_NORMAL_TO_CRACKLE[0]}/{BASELINE_NORMAL_TO_CRACKLE[1]}')
print(f'  This run — FULL test Normal->Crackle:               {full_cm[0,1]}/{full_cm[0].sum()}')
print(f'  This run — icbhi-only test Normal->Crackle:         {icbhi_cm[0,1]}/{icbhi_cm[0].sum()}')
print(f'  This run — hf_lung-only test Normal->Crackle:       {hf_cm[0,1]}/{hf_cm[0].sum()}')

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, (name, cm) in zip(axes, [('Full test', full_cm), ('ICBHI-only test', icbhi_cm),
                                  ('HF_Lung-only test', hf_cm)]):
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, xticklabels=SOUND_NAMES,
                yticklabels=SOUND_NAMES, cmap='Blues')
    ax.set_title(name); ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_test_confusion_matrices.png', dpi=150)
print(f'\nSaved test confusion matrices -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_test_confusion_matrices.png')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax = axes[0]
ax.plot(history.history['sound_accuracy'], label='Train Acc')
ax.plot(history.history['val_sound_accuracy'], label='Val Acc')
ax.set_title('Sound Accuracy'); ax.set_xlabel('Epoch'); ax.legend()

ax = axes[1]
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_title('Total Loss'); ax.set_xlabel('Epoch'); ax.legend()

ax = axes[2]
ax.plot(history.history['val_icbhi'], label='Val ICBHI', color='green')
ax.axhline(y=max(history.history['val_icbhi']), color='red', linestyle='--',
           label=f'Best: {max(history.history["val_icbhi"]):.4f}')
ax.set_title('ICBHI Score'); ax.set_xlabel('Epoch'); ax.legend()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_training_curves.png', dpi=150)
print(f'Saved training curves -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_training_curves.png')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cm_sound = confusion_matrix(y_sv, y_pred, labels=list(range(NUM_SOUND)))
sns.heatmap(cm_sound, annot=True, fmt='d', ax=axes[0],
            xticklabels=SOUND_NAMES, yticklabels=SOUND_NAMES, cmap='Blues')
axes[0].set_title('Sound Confusion Matrix')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

cm_diag = confusion_matrix(y_dv[diag_mask_val], y_pred_d[diag_mask_val], labels=list(range(NUM_DIAGNOSIS)))
sns.heatmap(cm_diag, annot=True, fmt='d', ax=axes[1],
            xticklabels=DIAG_NAMES, yticklabels=DIAG_NAMES, cmap='Oranges')
axes[1].set_title('Diagnosis Confusion Matrix')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_confusion_matrices.png', dpi=150)
print(f'Saved confusion matrices -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_confusion_matrices.png')

Overwriting scripts/train_multitask.py


In [ ]:
!python scripts/train_multitask.py

Train: 27810 | Val: 9106
Train distribution:
sound_label
0    12216
1     9977
2     5197
3      420
Name: count, dtype: int64
Loading val...
Val: (9106, 128, 63, 1)
Loading train...
Train: (27810, 128, 63, 1)
Diagnosis-labeled rows: train 2891/27810, val 799/9106
2026-08-31 17:50:24.057036: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1788198624.058502   18114 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Class weights: [0.5691306483300589, 0.6968527613511075, 1.3377910332884357, 16.553571428571427]
Source weight map: {'icbhi': 9.0, 'hf_lung': 1.0}
Sound sample_weight: min=0.569, max=148.982, mean=2.522

Starting training...
2026-08-31 17:50:26.538857: W external/local_xla/xla/tsl/framework/cpu_all

In [ ]:
import tensorflow as tf
import keras
print("TF:", tf.__version__)
print("Keras:", keras.__version__)

TF: 2.20.0
Keras: 3.13.2


In [ ]:
!grep -n "sample_weight=\[" scripts/train_multitask.py

grep: scripts/train_multitask.py: No such file or directory


In [ ]:
%%writefile scripts/train_multitask.py
"""
Final multitask training — baseline CNN architecture that works,
trained on original 2,891 samples with both sound + diagnosis heads.
Using original data (not augmented) to avoid overfitting collapse.
"""
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

SEED=42; NUM_SOUND=4; NUM_DIAGNOSIS=7; BATCH_SIZE=32
EPOCHS=80; LR=1e-3; LR_MIN=1e-6; TARGET_FRAMES=63; N_MELS=128
DIAG_WEIGHT=0.1; CKPT_DIR='data/checkpoints'; RESULTS_DIR='data/results'
DIAG_MISSING = -1
MANIFEST_PATH = 'data/processed/manifest_merged.csv'
RUN_SUFFIX = 'sourceweighted'
CKPT_NAME = f'multitask_{RUN_SUFFIX}_best.keras'
PREDICT_BATCH_SIZE = 128
BASELINE_ICBHI_SCORE = 0.6226
BASELINE_NORMAL_TO_CRACKLE = (171, 506)
UNWEIGHTED_COMBINED_SCORES = {'full': 0.7164, 'icbhi': 0.5196, 'hf_lung': 0.7608}
ICBHI_SOURCE_WEIGHT = 9.0
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
SOUND_NAMES = ['Normal', 'Crackle', 'Wheeze', 'Both']
DIAG_NAMES  = ['Healthy', 'COPD', 'URTI', 'Bronchiectasis', 'Pneumonia', 'Bronchiolitis', 'Other']


def pad_or_truncate(feat, t=TARGET_FRAMES):
    c = feat.shape[-1]
    if c < t:
        feat = np.pad(feat, [(0,0), (0, t-c)])
    else:
        feat = feat[..., :t]
    return feat


def focal_loss(gamma=2.0, class_weights=None):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce  = -y_true * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        focal = tf.pow(1.0 - p_t, gamma) * ce
        if class_weights is not None:
            weights = tf.reduce_sum(y_true * tf.constant(class_weights, dtype=tf.float32), axis=-1, keepdims=True)
            focal = focal * weights
        return tf.reduce_mean(focal, axis=-1)
    loss_fn.__name__ = 'focal_loss'
    return loss_fn


def icbhi_score(y_true, y_pred, n=NUM_SOUND):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    se, sp = [], []
    for i in range(n):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-tp-fn-fp
        se.append(tp/(tp+fn) if tp+fn>0 else 0.0)
        sp.append(tn/(tn+fp) if tn+fp>0 else 0.0)
    return (np.mean(se)+np.mean(sp))/2.0


class ICBHICallback(tf.keras.callbacks.Callback):
    def __init__(self, X, y):
        super().__init__(); self.X=X; self.y=y
        self.best = 0.0
    def on_epoch_end(self, epoch, logs=None):
        pred  = np.argmax(self.model.predict(self.X, verbose=0)[0], axis=1)
        score = icbhi_score(self.y, pred)
        logs['val_icbhi'] = score
        if score > self.best:
            self.best = score
            self.model.save(f'{CKPT_DIR}/{CKPT_NAME}')
            print(f'  val_icbhi: {score:.4f} *** NEW BEST — saved ***')
        else:
            print(f'  val_icbhi: {score:.4f}')


manifest = pd.read_csv(MANIFEST_PATH)
train_df  = manifest[manifest['split'] == 'train']
val_df    = manifest[manifest['split'] == 'val']
print(f'Train: {len(train_df)} | Val: {len(val_df)}')
print('Train distribution:')
print(train_df['sound_label'].value_counts().sort_index())


def load_split(df):
    X, ys, yd, src = [], [], [], []
    for _, row in df.iterrows():
        feat = np.load(row['features_path'])
        feat = pad_or_truncate(feat)
        if feat.ndim == 2:
            feat = feat[..., np.newaxis]
        X.append(feat); ys.append(row['sound_label']); yd.append(row['diagnosis_label'])
        src.append(row['source_dataset'])
    return (np.array(X, np.float32), np.array(ys, np.int32), np.array(yd, np.int32), np.array(src))


print('Loading val...')
X_val, y_sv, y_dv, src_val = load_split(val_df)
print(f'Val: {X_val.shape}')

print('Loading train...')
X_tr, y_str, y_dtr, src_tr = load_split(train_df)
print(f'Train: {X_tr.shape}')

y_str_oh = tf.keras.utils.to_categorical(y_str, NUM_SOUND)
y_sv_oh  = tf.keras.utils.to_categorical(y_sv,  NUM_SOUND)

diag_mask_train = (y_dtr != DIAG_MISSING)
diag_mask_val   = (y_dv  != DIAG_MISSING)

y_dtr_oh = tf.keras.utils.to_categorical(np.where(diag_mask_train, y_dtr, 0), NUM_DIAGNOSIS)
y_dv_oh  = tf.keras.utils.to_categorical(np.where(diag_mask_val,   y_dv,  0), NUM_DIAGNOSIS)
y_dtr_oh[~diag_mask_train] = 0.0
y_dv_oh[~diag_mask_val]    = 0.0

print(f'Diagnosis-labeled rows: train {diag_mask_train.sum()}/{len(diag_mask_train)}, '
      f'val {diag_mask_val.sum()}/{len(diag_mask_val)}')


def build_model():
    inp = tf.keras.Input(shape=(N_MELS, TARGET_FRAMES, 1))
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    shared = tf.keras.layers.Dense(256, activation='relu')(x)
    shared = tf.keras.layers.Dropout(0.5)(shared)

    s = tf.keras.layers.Dense(128, activation='relu')(shared)
    s = tf.keras.layers.Dropout(0.3)(s)
    sound_out = tf.keras.layers.Dense(NUM_SOUND, activation='softmax', name='sound')(s)

    d = tf.keras.layers.Dense(128, activation='relu')(shared)
    d = tf.keras.layers.Dropout(0.3)(d)
    diag_out = tf.keras.layers.Dense(NUM_DIAGNOSIS, activation='softmax', name='diagnosis')(d)

    return tf.keras.Model(inp, [sound_out, diag_out])


model = build_model()


def lr_schedule(epoch):
    cos = np.cos(np.pi * epoch / EPOCHS)
    return float(LR_MIN + 0.5*(LR - LR_MIN)*(1 + cos))


callbacks = [
    ICBHICallback(X_val, y_sv),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_icbhi', mode='max',
        patience=20, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.LearningRateScheduler(lr_schedule, verbose=0),
]

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y_str), y=y_str)
cw_list = [float(cw[i]) for i in range(NUM_SOUND)]
print(f'Class weights: {cw_list}')

SOURCE_WEIGHT_MAP = {'icbhi': ICBHI_SOURCE_WEIGHT, 'hf_lung': 1.0}
class_weight_per_sample  = np.array([cw_list[label] for label in y_str], dtype=np.float32)
source_weight_per_sample = np.array([SOURCE_WEIGHT_MAP[s] for s in src_tr], dtype=np.float32)
sound_sample_weight = class_weight_per_sample * source_weight_per_sample
print(f'Source weight map: {SOURCE_WEIGHT_MAP}')
print(f'Sound sample_weight: min={sound_sample_weight.min():.3f}, '
      f'max={sound_sample_weight.max():.3f}, mean={sound_sample_weight.mean():.3f}')

diag_sample_weight = np.ones(len(y_str), dtype=np.float32)

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss={'sound': focal_loss(4.0), 'diagnosis': focal_loss(2.0)},
    loss_weights={'sound': 1.0, 'diagnosis': DIAG_WEIGHT},
    metrics={'sound': 'accuracy', 'diagnosis': 'accuracy'}
)

print('\nStarting training...')
history = model.fit(
    X_tr, [y_str_oh, y_dtr_oh],
    sample_weight=[sound_sample_weight, diag_sample_weight],
    validation_data=(X_val, {'sound': y_sv_oh, 'diagnosis': y_dv_oh}),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print(f'\nBest val_icbhi: {max(history.history["val_icbhi"]):.4f}')
model_best = tf.keras.models.load_model(f'{CKPT_DIR}/{CKPT_NAME}', compile=False)
val_preds  = model_best.predict(X_val, batch_size=PREDICT_BATCH_SIZE, verbose=0)
y_pred     = np.argmax(val_preds[0], axis=1)
print('\nSound Classification Report:')
print(classification_report(y_sv, y_pred, target_names=SOUND_NAMES))
y_pred_d   = np.argmax(val_preds[1], axis=1)
print('Diagnosis Classification Report:')
print(classification_report(y_dv[diag_mask_val], y_pred_d[diag_mask_val],
                             labels=list(range(NUM_DIAGNOSIS)), target_names=DIAG_NAMES))


def load_split_with_source(df):
    X, ys, src = [], [], []
    for _, row in df.iterrows():
        feat = np.load(row['features_path'])
        feat = pad_or_truncate(feat)
        if feat.ndim == 2:
            feat = feat[..., np.newaxis]
        X.append(feat); ys.append(row['sound_label']); src.append(row['source_dataset'])
    return np.array(X, np.float32), np.array(ys, np.int32), np.array(src)


print('\nLoading test...')
test_df = manifest[manifest['split'] == 'test']
X_test, y_s_test, src_test = load_split_with_source(test_df)
n_icbhi_test, n_hf_test = (src_test == 'icbhi').sum(), (src_test == 'hf_lung').sum()
print(f'Test: {X_test.shape}  ({n_icbhi_test} icbhi, {n_hf_test} hf_lung)')

test_preds  = model_best.predict(X_test, batch_size=PREDICT_BATCH_SIZE, verbose=0)
y_pred_test = np.argmax(test_preds[0], axis=1)


def report_subset(name, mask):
    yt, yp = y_s_test[mask], y_pred_test[mask]
    score = icbhi_score(yt, yp)
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_SOUND)))
    print(f'\n=== TEST subset: {name} (n={mask.sum()}) ===')
    print(f'  ICBHI score: {score:.4f} ({score*100:.2f}%)')
    print(f'  Normal->Crackle confusions: {cm[0, 1]} / {cm[0].sum()} true Normal')
    print('  Confusion matrix (rows=true, cols=pred), order Normal/Crackle/Wheeze/Both:')
    print(cm)
    print(classification_report(yt, yp, target_names=SOUND_NAMES, zero_division=0))
    return score, cm


full_score, full_cm   = report_subset('FULL (icbhi+hf_lung)', np.ones(len(y_s_test), dtype=bool))
icbhi_score_, icbhi_cm = report_subset("source_dataset=='icbhi'", src_test == 'icbhi')
hf_score, hf_cm        = report_subset("source_dataset=='hf_lung'", src_test == 'hf_lung')

print('\n=== Three-way comparison: sem-6 baseline vs. unweighted combined vs. this (source-weighted) run ===')
print('  NOTE: sem-6 baseline (62.26%, 171/506) was measured on the ORIGINAL ICBHI-only')
print('  VAL split (799 rows). The unweighted-combined and this-run numbers are both on')
print('  the (larger, merged) TEST split.')
print(f"  {'subset':22s} {'sem-6 baseline':>15s} {'unweighted combined':>21s} {'this run (source-wt)':>22s}")
print(f"  {'FULL (icbhi+hf_lung)':22s} {'—':>15s} "
      f"{UNWEIGHTED_COMBINED_SCORES['full']*100:20.2f}% {full_score*100:21.2f}%")
print(f"  {'icbhi-only':22s} {BASELINE_ICBHI_SCORE*100:14.2f}% "
      f"{UNWEIGHTED_COMBINED_SCORES['icbhi']*100:20.2f}% {icbhi_score_*100:21.2f}%")
print(f"  {'hf_lung-only':22s} {'—':>15s} "
      f"{UNWEIGHTED_COMBINED_SCORES['hf_lung']*100:20.2f}% {hf_score*100:21.2f}%")
print()
print(f"  icbhi-only delta vs sem-6 baseline:        {(icbhi_score_-BASELINE_ICBHI_SCORE)*100:+.2f} pts")
print(f"  icbhi-only delta vs unweighted combined:   {(icbhi_score_-UNWEIGHTED_COMBINED_SCORES['icbhi'])*100:+.2f} pts")
print(f"  full delta vs unweighted combined:         {(full_score-UNWEIGHTED_COMBINED_SCORES['full'])*100:+.2f} pts")
print(f"  hf_lung-only delta vs unweighted combined: {(hf_score-UNWEIGHTED_COMBINED_SCORES['hf_lung'])*100:+.2f} pts")
print()
print(f'  Baseline Normal->Crackle (val, icbhi-only, sem-6): {BASELINE_NORMAL_TO_CRACKLE[0]}/{BASELINE_NORMAL_TO_CRACKLE[1]}')
print(f'  This run — FULL test Normal->Crackle:               {full_cm[0,1]}/{full_cm[0].sum()}')
print(f'  This run — icbhi-only test Normal->Crackle:         {icbhi_cm[0,1]}/{icbhi_cm[0].sum()}')
print(f'  This run — hf_lung-only test Normal->Crackle:       {hf_cm[0,1]}/{hf_cm[0].sum()}')

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, (name, cm) in zip(axes, [('Full test', full_cm), ('ICBHI-only test', icbhi_cm),
                                  ('HF_Lung-only test', hf_cm)]):
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, xticklabels=SOUND_NAMES,
                yticklabels=SOUND_NAMES, cmap='Blues')
    ax.set_title(name); ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_test_confusion_matrices.png', dpi=150)
print(f'\nSaved test confusion matrices -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_test_confusion_matrices.png')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax = axes[0]
ax.plot(history.history['sound_accuracy'], label='Train Acc')
ax.plot(history.history['val_sound_accuracy'], label='Val Acc')
ax.set_title('Sound Accuracy'); ax.set_xlabel('Epoch'); ax.legend()

ax = axes[1]
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_title('Total Loss'); ax.set_xlabel('Epoch'); ax.legend()

ax = axes[2]
ax.plot(history.history['val_icbhi'], label='Val ICBHI', color='green')
ax.axhline(y=max(history.history['val_icbhi']), color='red', linestyle='--',
           label=f'Best: {max(history.history["val_icbhi"]):.4f}')
ax.set_title('ICBHI Score'); ax.set_xlabel('Epoch'); ax.legend()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_training_curves.png', dpi=150)
print(f'Saved training curves -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_training_curves.png')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cm_sound = confusion_matrix(y_sv, y_pred, labels=list(range(NUM_SOUND)))
sns.heatmap(cm_sound, annot=True, fmt='d', ax=axes[0],
            xticklabels=SOUND_NAMES, yticklabels=SOUND_NAMES, cmap='Blues')
axes[0].set_title('Sound Confusion Matrix')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

cm_diag = confusion_matrix(y_dv[diag_mask_val], y_pred_d[diag_mask_val], labels=list(range(NUM_DIAGNOSIS)))
sns.heatmap(cm_diag, annot=True, fmt='d', ax=axes[1],
            xticklabels=DIAG_NAMES, yticklabels=DIAG_NAMES, cmap='Oranges')
axes[1].set_title('Diagnosis Confusion Matrix')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_confusion_matrices.png', dpi=150)
print(f'Saved confusion matrices -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_confusion_matrices.png')

Writing scripts/train_multitask.py


FileNotFoundError: [Errno 2] No such file or directory: 'scripts/train_multitask.py'

In [ ]:
%cd /content/project
!pwd
!ls

[Errno 2] No such file or directory: '/content/project'
/content
/content
sample_data


In [ ]:
import pandas as pd
import re

manifest = pd.read_csv('data/processed/manifest_merged.csv')

def fix_path(p):
    if pd.isna(p):
        return p
    p = str(p)
    match = re.search(r'[Dd]ata[\\/].*', p)
    if match:
        relative = match.group(0).replace('\\', '/')
        return '/content/project/' + relative
    return p

manifest['features_path'] = manifest['features_path'].apply(fix_path)
manifest['audio_path'] = manifest['audio_path'].apply(fix_path)

import os
missing = manifest['features_path'].apply(lambda p: not os.path.exists(p)).sum()
print(f"Missing files after fix: {missing} / {len(manifest)}")

if missing == 0:
    manifest.to_csv('data/processed/manifest_merged.csv', index=False)
    print("Saved corrected manifest.")
else:
    print("DO NOT proceed — investigate missing paths.")

Missing files after fix: 0 / 46852
Saved corrected manifest.


In [ ]:
%%writefile scripts/train_multitask.py
"""
Final multitask training — baseline CNN architecture that works,
trained on original 2,891 samples with both sound + diagnosis heads.
Using original data (not augmented) to avoid overfitting collapse.
"""
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

SEED=42; NUM_SOUND=4; NUM_DIAGNOSIS=7; BATCH_SIZE=32
EPOCHS=80; LR=1e-3; LR_MIN=1e-6; TARGET_FRAMES=63; N_MELS=128
DIAG_WEIGHT=0.1; CKPT_DIR='data/checkpoints'; RESULTS_DIR='data/results'
DIAG_MISSING = -1
MANIFEST_PATH = 'data/processed/manifest_merged.csv'
RUN_SUFFIX = 'sourceweighted'
CKPT_NAME = f'multitask_{RUN_SUFFIX}_best.keras'
PREDICT_BATCH_SIZE = 128
BASELINE_ICBHI_SCORE = 0.6226
BASELINE_NORMAL_TO_CRACKLE = (171, 506)
UNWEIGHTED_COMBINED_SCORES = {'full': 0.7164, 'icbhi': 0.5196, 'hf_lung': 0.7608}
ICBHI_SOURCE_WEIGHT = 9.0
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
SOUND_NAMES = ['Normal', 'Crackle', 'Wheeze', 'Both']
DIAG_NAMES  = ['Healthy', 'COPD', 'URTI', 'Bronchiectasis', 'Pneumonia', 'Bronchiolitis', 'Other']


def pad_or_truncate(feat, t=TARGET_FRAMES):
    c = feat.shape[-1]
    if c < t:
        feat = np.pad(feat, [(0,0), (0, t-c)])
    else:
        feat = feat[..., :t]
    return feat


def focal_loss(gamma=2.0, class_weights=None):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce  = -y_true * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        focal = tf.pow(1.0 - p_t, gamma) * ce
        if class_weights is not None:
            weights = tf.reduce_sum(y_true * tf.constant(class_weights, dtype=tf.float32), axis=-1, keepdims=True)
            focal = focal * weights
        return tf.reduce_mean(focal, axis=-1)
    loss_fn.__name__ = 'focal_loss'
    return loss_fn


def icbhi_score(y_true, y_pred, n=NUM_SOUND):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    se, sp = [], []
    for i in range(n):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-tp-fn-fp
        se.append(tp/(tp+fn) if tp+fn>0 else 0.0)
        sp.append(tn/(tn+fp) if tn+fp>0 else 0.0)
    return (np.mean(se)+np.mean(sp))/2.0


class ICBHICallback(tf.keras.callbacks.Callback):
    def __init__(self, X, y):
        super().__init__(); self.X=X; self.y=y
        self.best = 0.0
    def on_epoch_end(self, epoch, logs=None):
        pred  = np.argmax(self.model.predict(self.X, verbose=0)[0], axis=1)
        score = icbhi_score(self.y, pred)
        logs['val_icbhi'] = score
        if score > self.best:
            self.best = score
            self.model.save(f'{CKPT_DIR}/{CKPT_NAME}')
            print(f'  val_icbhi: {score:.4f} *** NEW BEST — saved ***')
        else:
            print(f'  val_icbhi: {score:.4f}')


manifest = pd.read_csv(MANIFEST_PATH)
train_df  = manifest[manifest['split'] == 'train']
val_df    = manifest[manifest['split'] == 'val']
print(f'Train: {len(train_df)} | Val: {len(val_df)}')
print('Train distribution:')
print(train_df['sound_label'].value_counts().sort_index())


def load_split(df):
    X, ys, yd, src = [], [], [], []
    for _, row in df.iterrows():
        feat = np.load(row['features_path'])
        feat = pad_or_truncate(feat)
        if feat.ndim == 2:
            feat = feat[..., np.newaxis]
        X.append(feat); ys.append(row['sound_label']); yd.append(row['diagnosis_label'])
        src.append(row['source_dataset'])
    return (np.array(X, np.float32), np.array(ys, np.int32), np.array(yd, np.int32), np.array(src))


print('Loading val...')
X_val, y_sv, y_dv, src_val = load_split(val_df)
print(f'Val: {X_val.shape}')

print('Loading train...')
X_tr, y_str, y_dtr, src_tr = load_split(train_df)
print(f'Train: {X_tr.shape}')

y_str_oh = tf.keras.utils.to_categorical(y_str, NUM_SOUND)
y_sv_oh  = tf.keras.utils.to_categorical(y_sv,  NUM_SOUND)

diag_mask_train = (y_dtr != DIAG_MISSING)
diag_mask_val   = (y_dv  != DIAG_MISSING)

y_dtr_oh = tf.keras.utils.to_categorical(np.where(diag_mask_train, y_dtr, 0), NUM_DIAGNOSIS)
y_dv_oh  = tf.keras.utils.to_categorical(np.where(diag_mask_val,   y_dv,  0), NUM_DIAGNOSIS)
y_dtr_oh[~diag_mask_train] = 0.0
y_dv_oh[~diag_mask_val]    = 0.0

print(f'Diagnosis-labeled rows: train {diag_mask_train.sum()}/{len(diag_mask_train)}, '
      f'val {diag_mask_val.sum()}/{len(diag_mask_val)}')


def build_model():
    inp = tf.keras.Input(shape=(N_MELS, TARGET_FRAMES, 1))
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    shared = tf.keras.layers.Dense(256, activation='relu')(x)
    shared = tf.keras.layers.Dropout(0.5)(shared)

    s = tf.keras.layers.Dense(128, activation='relu')(shared)
    s = tf.keras.layers.Dropout(0.3)(s)
    sound_out = tf.keras.layers.Dense(NUM_SOUND, activation='softmax', name='sound')(s)

    d = tf.keras.layers.Dense(128, activation='relu')(shared)
    d = tf.keras.layers.Dropout(0.3)(d)
    diag_out = tf.keras.layers.Dense(NUM_DIAGNOSIS, activation='softmax', name='diagnosis')(d)

    return tf.keras.Model(inp, [sound_out, diag_out])


model = build_model()


def lr_schedule(epoch):
    cos = np.cos(np.pi * epoch / EPOCHS)
    return float(LR_MIN + 0.5*(LR - LR_MIN)*(1 + cos))


callbacks = [
    ICBHICallback(X_val, y_sv),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_icbhi', mode='max',
        patience=20, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.LearningRateScheduler(lr_schedule, verbose=0),
]

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y_str), y=y_str)
cw_list = [float(cw[i]) for i in range(NUM_SOUND)]
print(f'Class weights: {cw_list}')

SOURCE_WEIGHT_MAP = {'icbhi': ICBHI_SOURCE_WEIGHT, 'hf_lung': 1.0}
class_weight_per_sample  = np.array([cw_list[label] for label in y_str], dtype=np.float32)
source_weight_per_sample = np.array([SOURCE_WEIGHT_MAP[s] for s in src_tr], dtype=np.float32)
sound_sample_weight = class_weight_per_sample * source_weight_per_sample
print(f'Source weight map: {SOURCE_WEIGHT_MAP}')
print(f'Sound sample_weight: min={sound_sample_weight.min():.3f}, '
      f'max={sound_sample_weight.max():.3f}, mean={sound_sample_weight.mean():.3f}')

diag_sample_weight = np.ones(len(y_str), dtype=np.float32)

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss={'sound': focal_loss(4.0), 'diagnosis': focal_loss(2.0)},
    loss_weights={'sound': 1.0, 'diagnosis': DIAG_WEIGHT},
    metrics={'sound': 'accuracy', 'diagnosis': 'accuracy'}
)

print('\nStarting training...')
history = model.fit(
    X_tr, [y_str_oh, y_dtr_oh],
    sample_weight=[sound_sample_weight, diag_sample_weight],
    validation_data=(X_val, {'sound': y_sv_oh, 'diagnosis': y_dv_oh}),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print(f'\nBest val_icbhi: {max(history.history["val_icbhi"]):.4f}')
model_best = tf.keras.models.load_model(f'{CKPT_DIR}/{CKPT_NAME}', compile=False)
val_preds  = model_best.predict(X_val, batch_size=PREDICT_BATCH_SIZE, verbose=0)
y_pred     = np.argmax(val_preds[0], axis=1)
print('\nSound Classification Report:')
print(classification_report(y_sv, y_pred, target_names=SOUND_NAMES))
y_pred_d   = np.argmax(val_preds[1], axis=1)
print('Diagnosis Classification Report:')
print(classification_report(y_dv[diag_mask_val], y_pred_d[diag_mask_val],
                             labels=list(range(NUM_DIAGNOSIS)), target_names=DIAG_NAMES))


def load_split_with_source(df):
    X, ys, src = [], [], []
    for _, row in df.iterrows():
        feat = np.load(row['features_path'])
        feat = pad_or_truncate(feat)
        if feat.ndim == 2:
            feat = feat[..., np.newaxis]
        X.append(feat); ys.append(row['sound_label']); src.append(row['source_dataset'])
    return np.array(X, np.float32), np.array(ys, np.int32), np.array(src)


print('\nLoading test...')
test_df = manifest[manifest['split'] == 'test']
X_test, y_s_test, src_test = load_split_with_source(test_df)
n_icbhi_test, n_hf_test = (src_test == 'icbhi').sum(), (src_test == 'hf_lung').sum()
print(f'Test: {X_test.shape}  ({n_icbhi_test} icbhi, {n_hf_test} hf_lung)')

test_preds  = model_best.predict(X_test, batch_size=PREDICT_BATCH_SIZE, verbose=0)
y_pred_test = np.argmax(test_preds[0], axis=1)


def report_subset(name, mask):
    yt, yp = y_s_test[mask], y_pred_test[mask]
    score = icbhi_score(yt, yp)
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_SOUND)))
    print(f'\n=== TEST subset: {name} (n={mask.sum()}) ===')
    print(f'  ICBHI score: {score:.4f} ({score*100:.2f}%)')
    print(f'  Normal->Crackle confusions: {cm[0, 1]} / {cm[0].sum()} true Normal')
    print('  Confusion matrix (rows=true, cols=pred), order Normal/Crackle/Wheeze/Both:')
    print(cm)
    print(classification_report(yt, yp, target_names=SOUND_NAMES, zero_division=0))
    return score, cm


full_score, full_cm   = report_subset('FULL (icbhi+hf_lung)', np.ones(len(y_s_test), dtype=bool))
icbhi_score_, icbhi_cm = report_subset("source_dataset=='icbhi'", src_test == 'icbhi')
hf_score, hf_cm        = report_subset("source_dataset=='hf_lung'", src_test == 'hf_lung')

print('\n=== Three-way comparison: sem-6 baseline vs. unweighted combined vs. this (source-weighted) run ===')
print('  NOTE: sem-6 baseline (62.26%, 171/506) was measured on the ORIGINAL ICBHI-only')
print('  VAL split (799 rows). The unweighted-combined and this-run numbers are both on')
print('  the (larger, merged) TEST split.')
print(f"  {'subset':22s} {'sem-6 baseline':>15s} {'unweighted combined':>21s} {'this run (source-wt)':>22s}")
print(f"  {'FULL (icbhi+hf_lung)':22s} {'—':>15s} "
      f"{UNWEIGHTED_COMBINED_SCORES['full']*100:20.2f}% {full_score*100:21.2f}%")
print(f"  {'icbhi-only':22s} {BASELINE_ICBHI_SCORE*100:14.2f}% "
      f"{UNWEIGHTED_COMBINED_SCORES['icbhi']*100:20.2f}% {icbhi_score_*100:21.2f}%")
print(f"  {'hf_lung-only':22s} {'—':>15s} "
      f"{UNWEIGHTED_COMBINED_SCORES['hf_lung']*100:20.2f}% {hf_score*100:21.2f}%")
print()
print(f"  icbhi-only delta vs sem-6 baseline:        {(icbhi_score_-BASELINE_ICBHI_SCORE)*100:+.2f} pts")
print(f"  icbhi-only delta vs unweighted combined:   {(icbhi_score_-UNWEIGHTED_COMBINED_SCORES['icbhi'])*100:+.2f} pts")
print(f"  full delta vs unweighted combined:         {(full_score-UNWEIGHTED_COMBINED_SCORES['full'])*100:+.2f} pts")
print(f"  hf_lung-only delta vs unweighted combined: {(hf_score-UNWEIGHTED_COMBINED_SCORES['hf_lung'])*100:+.2f} pts")
print()
print(f'  Baseline Normal->Crackle (val, icbhi-only, sem-6): {BASELINE_NORMAL_TO_CRACKLE[0]}/{BASELINE_NORMAL_TO_CRACKLE[1]}')
print(f'  This run — FULL test Normal->Crackle:               {full_cm[0,1]}/{full_cm[0].sum()}')
print(f'  This run — icbhi-only test Normal->Crackle:         {icbhi_cm[0,1]}/{icbhi_cm[0].sum()}')
print(f'  This run — hf_lung-only test Normal->Crackle:       {hf_cm[0,1]}/{hf_cm[0].sum()}')

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, (name, cm) in zip(axes, [('Full test', full_cm), ('ICBHI-only test', icbhi_cm),
                                  ('HF_Lung-only test', hf_cm)]):
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, xticklabels=SOUND_NAMES,
                yticklabels=SOUND_NAMES, cmap='Blues')
    ax.set_title(name); ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_test_confusion_matrices.png', dpi=150)
print(f'\nSaved test confusion matrices -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_test_confusion_matrices.png')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax = axes[0]
ax.plot(history.history['sound_accuracy'], label='Train Acc')
ax.plot(history.history['val_sound_accuracy'], label='Val Acc')
ax.set_title('Sound Accuracy'); ax.set_xlabel('Epoch'); ax.legend()

ax = axes[1]
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_title('Total Loss'); ax.set_xlabel('Epoch'); ax.legend()

ax = axes[2]
ax.plot(history.history['val_icbhi'], label='Val ICBHI', color='green')
ax.axhline(y=max(history.history['val_icbhi']), color='red', linestyle='--',
           label=f'Best: {max(history.history["val_icbhi"]):.4f}')
ax.set_title('ICBHI Score'); ax.set_xlabel('Epoch'); ax.legend()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_training_curves.png', dpi=150)
print(f'Saved training curves -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_training_curves.png')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cm_sound = confusion_matrix(y_sv, y_pred, labels=list(range(NUM_SOUND)))
sns.heatmap(cm_sound, annot=True, fmt='d', ax=axes[0],
            xticklabels=SOUND_NAMES, yticklabels=SOUND_NAMES, cmap='Blues')
axes[0].set_title('Sound Confusion Matrix')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

cm_diag = confusion_matrix(y_dv[diag_mask_val], y_pred_d[diag_mask_val], labels=list(range(NUM_DIAGNOSIS)))
sns.heatmap(cm_diag, annot=True, fmt='d', ax=axes[1],
            xticklabels=DIAG_NAMES, yticklabels=DIAG_NAMES, cmap='Oranges')
axes[1].set_title('Diagnosis Confusion Matrix')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_confusion_matrices.png', dpi=150)
print(f'Saved confusion matrices -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_confusion_matrices.png')

Overwriting scripts/train_multitask.py


In [ ]:
!python scripts/train_multitask.py

Train: 27810 | Val: 9106
Train distribution:
sound_label
0    12216
1     9977
2     5197
3      420
Name: count, dtype: int64
Loading val...
Val: (9106, 128, 63, 1)
Loading train...
Train: (27810, 128, 63, 1)
Diagnosis-labeled rows: train 2891/27810, val 799/9106
2026-08-31 20:15:00.723059: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1788207300.724506    8227 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Class weights: [0.5691306483300589, 0.6968527613511075, 1.3377910332884357, 16.553571428571427]
Source weight map: {'icbhi': 9.0, 'hf_lung': 1.0}
Sound sample_weight: min=0.569, max=148.982, mean=2.522

Starting training...
2026-08-31 20:15:03.267426: W external/local_xla/xla/tsl/framework/cpu_all

In [ ]:
%cd /content/project
!ls

/content/project
data  scripts  src


In [ ]:
%%writefile scripts/train_multitask.py
"""
Final multitask training — baseline CNN architecture that works,
trained on original 2,891 samples with both sound + diagnosis heads.
Using original data (not augmented) to avoid overfitting collapse.
"""
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

SEED=42; NUM_SOUND=4; NUM_DIAGNOSIS=7; BATCH_SIZE=32
EPOCHS=80; LR=1e-3; LR_MIN=1e-6; TARGET_FRAMES=63; N_MELS=128
DIAG_WEIGHT=0.1; CKPT_DIR='data/checkpoints'; RESULTS_DIR='data/results'
DIAG_MISSING = -1
MANIFEST_PATH = 'data/processed/manifest_merged.csv'
RUN_SUFFIX = 'batchstratified'
CKPT_NAME = f'multitask_{RUN_SUFFIX}_best.keras'
PREDICT_BATCH_SIZE = 128
BASELINE_ICBHI_SCORE = 0.6226
BASELINE_NORMAL_TO_CRACKLE = (171, 506)
UNWEIGHTED_COMBINED_SCORES = {'full': 0.7164, 'icbhi': 0.5196, 'hf_lung': 0.7608}
SOURCE_WEIGHTED_SCORES = {'full': 0.6537, 'icbhi': 0.5205, 'hf_lung': 0.6670}
ICBHI_BATCH_FRACTION = 0.3
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
SOUND_NAMES = ['Normal', 'Crackle', 'Wheeze', 'Both']
DIAG_NAMES  = ['Healthy', 'COPD', 'URTI', 'Bronchiectasis', 'Pneumonia', 'Bronchiolitis', 'Other']


def pad_or_truncate(feat, t=TARGET_FRAMES):
    c = feat.shape[-1]
    if c < t:
        feat = np.pad(feat, [(0,0), (0, t-c)])
    else:
        feat = feat[..., :t]
    return feat


def focal_loss(gamma=2.0, class_weights=None):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce  = -y_true * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        focal = tf.pow(1.0 - p_t, gamma) * ce
        if class_weights is not None:
            weights = tf.reduce_sum(y_true * tf.constant(class_weights, dtype=tf.float32), axis=-1, keepdims=True)
            focal = focal * weights
        return tf.reduce_mean(focal, axis=-1)
    loss_fn.__name__ = 'focal_loss'
    return loss_fn


def icbhi_score(y_true, y_pred, n=NUM_SOUND):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    se, sp = [], []
    for i in range(n):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-tp-fn-fp
        se.append(tp/(tp+fn) if tp+fn>0 else 0.0)
        sp.append(tn/(tn+fp) if tn+fp>0 else 0.0)
    return (np.mean(se)+np.mean(sp))/2.0


class ICBHICallback(tf.keras.callbacks.Callback):
    def __init__(self, X, y):
        super().__init__(); self.X=X; self.y=y
        self.best = 0.0
    def on_epoch_end(self, epoch, logs=None):
        pred  = np.argmax(self.model.predict(self.X, verbose=0)[0], axis=1)
        score = icbhi_score(self.y, pred)
        logs['val_icbhi'] = score
        if score > self.best:
            self.best = score
            self.model.save(f'{CKPT_DIR}/{CKPT_NAME}')
            print(f'  val_icbhi: {score:.4f} *** NEW BEST — saved ***')
        else:
            print(f'  val_icbhi: {score:.4f}')


manifest = pd.read_csv(MANIFEST_PATH)
train_df  = manifest[manifest['split'] == 'train']
val_df    = manifest[manifest['split'] == 'val']
print(f'Train: {len(train_df)} | Val: {len(val_df)}')
print('Train distribution:')
print(train_df['sound_label'].value_counts().sort_index())


def load_split(df):
    X, ys, yd, src = [], [], [], []
    for _, row in df.iterrows():
        feat = np.load(row['features_path'])
        feat = pad_or_truncate(feat)
        if feat.ndim == 2:
            feat = feat[..., np.newaxis]
        X.append(feat); ys.append(row['sound_label']); yd.append(row['diagnosis_label'])
        src.append(row['source_dataset'])
    return (np.array(X, np.float32), np.array(ys, np.int32), np.array(yd, np.int32), np.array(src))


print('Loading val...')
X_val, y_sv, y_dv, src_val = load_split(val_df)
print(f'Val: {X_val.shape}')

print('Loading train...')
X_tr, y_str, y_dtr, src_tr = load_split(train_df)
print(f'Train: {X_tr.shape}')

y_str_oh = tf.keras.utils.to_categorical(y_str, NUM_SOUND)
y_sv_oh  = tf.keras.utils.to_categorical(y_sv,  NUM_SOUND)

diag_mask_train = (y_dtr != DIAG_MISSING)
diag_mask_val   = (y_dv  != DIAG_MISSING)

y_dtr_oh = tf.keras.utils.to_categorical(np.where(diag_mask_train, y_dtr, 0), NUM_DIAGNOSIS)
y_dv_oh  = tf.keras.utils.to_categorical(np.where(diag_mask_val,   y_dv,  0), NUM_DIAGNOSIS)
y_dtr_oh[~diag_mask_train] = 0.0
y_dv_oh[~diag_mask_val]    = 0.0

print(f'Diagnosis-labeled rows: train {diag_mask_train.sum()}/{len(diag_mask_train)}, '
      f'val {diag_mask_val.sum()}/{len(diag_mask_val)}')


def build_model():
    inp = tf.keras.Input(shape=(N_MELS, TARGET_FRAMES, 1))
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    shared = tf.keras.layers.Dense(256, activation='relu')(x)
    shared = tf.keras.layers.Dropout(0.5)(shared)

    s = tf.keras.layers.Dense(128, activation='relu')(shared)
    s = tf.keras.layers.Dropout(0.3)(s)
    sound_out = tf.keras.layers.Dense(NUM_SOUND, activation='softmax', name='sound')(s)

    d = tf.keras.layers.Dense(128, activation='relu')(shared)
    d = tf.keras.layers.Dropout(0.3)(d)
    diag_out = tf.keras.layers.Dense(NUM_DIAGNOSIS, activation='softmax', name='diagnosis')(d)

    return tf.keras.Model(inp, [sound_out, diag_out])


model = build_model()


def lr_schedule(epoch):
    cos = np.cos(np.pi * epoch / EPOCHS)
    return float(LR_MIN + 0.5*(LR - LR_MIN)*(1 + cos))


callbacks = [
    ICBHICallback(X_val, y_sv),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_icbhi', mode='max',
        patience=20, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.LearningRateScheduler(lr_schedule, verbose=0),
]

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y_str), y=y_str)
cw_list = [float(cw[i]) for i in range(NUM_SOUND)]
print(f'Class weights: {cw_list}')

sound_sample_weight = np.array([cw_list[label] for label in y_str], dtype=np.float32)
print(f'Sound sample_weight (class-weight only, no source multiplier): '
      f'min={sound_sample_weight.min():.3f}, max={sound_sample_weight.max():.3f}, '
      f'mean={sound_sample_weight.mean():.3f}')

diag_sample_weight = np.ones(len(y_str), dtype=np.float32)

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss={'sound': focal_loss(4.0), 'diagnosis': focal_loss(2.0)},
    loss_weights={'sound': 1.0, 'diagnosis': DIAG_WEIGHT},
    metrics={'sound': 'accuracy', 'diagnosis': 'accuracy'}
)

icbhi_mask = (src_tr == 'icbhi')
hf_mask    = (src_tr == 'hf_lung')
icbhi_batch_size = round(BATCH_SIZE * ICBHI_BATCH_FRACTION)
hf_batch_size    = BATCH_SIZE - icbhi_batch_size
print(f'Stratified batches: {icbhi_batch_size} icbhi + {hf_batch_size} hf_lung per '
      f'{BATCH_SIZE}-sample batch (realized fraction {icbhi_batch_size/BATCH_SIZE:.4f}, '
      f'target {ICBHI_BATCH_FRACTION}) -- pools: {icbhi_mask.sum()} icbhi / {hf_mask.sum()} hf_lung rows')


def make_pool_dataset(ids, batch_size, seed):
    ds = tf.data.Dataset.from_tensor_slices(ids)
    ds = ds.shuffle(buffer_size=len(ids), seed=seed, reshuffle_each_iteration=True)
    ds = ds.repeat()
    return ds.batch(batch_size, drop_remainder=True)


def concat_pool_ids(icbhi_ids, hf_ids):
    return tf.concat([icbhi_ids, hf_ids], axis=0)


icbhi_pool_ds = make_pool_dataset(np.where(icbhi_mask)[0], icbhi_batch_size, seed=SEED)
hf_pool_ds    = make_pool_dataset(np.where(hf_mask)[0],    hf_batch_size,    seed=SEED + 1)
batch_id_ds = tf.data.Dataset.zip((icbhi_pool_ds, hf_pool_ds)).map(
    concat_pool_ids, num_parallel_calls=tf.data.AUTOTUNE)

_X_tr_t   = tf.constant(X_tr)
_y_s_t    = tf.constant(y_str_oh)
_y_d_t    = tf.constant(y_dtr_oh)
_sw_s_t   = tf.constant(sound_sample_weight)
_sw_d_t   = tf.constant(diag_sample_weight)


def gather_batch(idx):
    x  = tf.gather(_X_tr_t, idx)
    ys = tf.gather(_y_s_t, idx)
    yd = tf.gather(_y_d_t, idx)
    sws = tf.gather(_sw_s_t, idx)
    swd = tf.gather(_sw_d_t, idx)
    return x, (ys, yd), (sws, swd)


train_ds = (batch_id_ds
            .map(gather_batch, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))

steps_per_epoch = -(-len(train_df) // BATCH_SIZE)
print(f'steps_per_epoch = {steps_per_epoch} (matches ceil({len(train_df)}/{BATCH_SIZE}))')

print('\nStarting training...')
history = model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    validation_data=(X_val, {'sound': y_sv_oh, 'diagnosis': y_dv_oh}),
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print(f'\nBest val_icbhi: {max(history.history["val_icbhi"]):.4f}')
model_best = tf.keras.models.load_model(f'{CKPT_DIR}/{CKPT_NAME}', compile=False)
val_preds  = model_best.predict(X_val, batch_size=PREDICT_BATCH_SIZE, verbose=0)
y_pred     = np.argmax(val_preds[0], axis=1)
print('\nSound Classification Report:')
print(classification_report(y_sv, y_pred, target_names=SOUND_NAMES))
y_pred_d   = np.argmax(val_preds[1], axis=1)
print('Diagnosis Classification Report:')
print(classification_report(y_dv[diag_mask_val], y_pred_d[diag_mask_val],
                             labels=list(range(NUM_DIAGNOSIS)), target_names=DIAG_NAMES))


def load_split_with_source(df):
    X, ys, src = [], [], []
    for _, row in df.iterrows():
        feat = np.load(row['features_path'])
        feat = pad_or_truncate(feat)
        if feat.ndim == 2:
            feat = feat[..., np.newaxis]
        X.append(feat); ys.append(row['sound_label']); src.append(row['source_dataset'])
    return np.array(X, np.float32), np.array(ys, np.int32), np.array(src)


print('\nLoading test...')
test_df = manifest[manifest['split'] == 'test']
X_test, y_s_test, src_test = load_split_with_source(test_df)
n_icbhi_test, n_hf_test = (src_test == 'icbhi').sum(), (src_test == 'hf_lung').sum()
print(f'Test: {X_test.shape}  ({n_icbhi_test} icbhi, {n_hf_test} hf_lung)')

test_preds  = model_best.predict(X_test, batch_size=PREDICT_BATCH_SIZE, verbose=0)
y_pred_test = np.argmax(test_preds[0], axis=1)


def report_subset(name, mask):
    yt, yp = y_s_test[mask], y_pred_test[mask]
    score = icbhi_score(yt, yp)
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_SOUND)))
    print(f'\n=== TEST subset: {name} (n={mask.sum()}) ===')
    print(f'  ICBHI score: {score:.4f} ({score*100:.2f}%)')
    print(f'  Normal->Crackle confusions: {cm[0, 1]} / {cm[0].sum()} true Normal')
    print('  Confusion matrix (rows=true, cols=pred), order Normal/Crackle/Wheeze/Both:')
    print(cm)
    print(classification_report(yt, yp, target_names=SOUND_NAMES, zero_division=0))
    return score, cm


full_score, full_cm   = report_subset('FULL (icbhi+hf_lung)', np.ones(len(y_s_test), dtype=bool))
icbhi_score_, icbhi_cm = report_subset("source_dataset=='icbhi'", src_test == 'icbhi')
hf_score, hf_cm        = report_subset("source_dataset=='hf_lung'", src_test == 'hf_lung')

print('\n=== Four-way comparison: sem-6 baseline / unweighted combined / source-weighted (failed) / batch-stratified (this run) ===')
print('  NOTE: sem-6 baseline (62.26%, 171/506) was measured on the ORIGINAL ICBHI-only')
print('  VAL split (799 rows). The other three columns are all on the (larger, merged) TEST split.')
col = "  {:22s} {:>15s} {:>21s} {:>19s} {:>22s}"
print(col.format('subset', 'sem-6 baseline', 'unweighted comb.', 'source-wt (failed)', 'batch-strat (this run)'))
print(col.format('FULL (icbhi+hf_lung)', '—',
                  f"{UNWEIGHTED_COMBINED_SCORES['full']*100:.2f}%",
                  f"{SOURCE_WEIGHTED_SCORES['full']*100:.2f}%",
                  f"{full_score*100:.2f}%"))
print(col.format('icbhi-only', f"{BASELINE_ICBHI_SCORE*100:.2f}%",
                  f"{UNWEIGHTED_COMBINED_SCORES['icbhi']*100:.2f}%",
                  f"{SOURCE_WEIGHTED_SCORES['icbhi']*100:.2f}%",
                  f"{icbhi_score_*100:.2f}%"))
print(col.format('hf_lung-only', '—',
                  f"{UNWEIGHTED_COMBINED_SCORES['hf_lung']*100:.2f}%",
                  f"{SOURCE_WEIGHTED_SCORES['hf_lung']*100:.2f}%",
                  f"{hf_score*100:.2f}%"))
print()
print(f"  icbhi-only delta vs sem-6 baseline:            {(icbhi_score_-BASELINE_ICBHI_SCORE)*100:+.2f} pts")
print(f"  icbhi-only delta vs unweighted combined:       {(icbhi_score_-UNWEIGHTED_COMBINED_SCORES['icbhi'])*100:+.2f} pts")
print(f"  icbhi-only delta vs source-weighted (failed):  {(icbhi_score_-SOURCE_WEIGHTED_SCORES['icbhi'])*100:+.2f} pts")
print(f"  full delta vs unweighted combined:             {(full_score-UNWEIGHTED_COMBINED_SCORES['full'])*100:+.2f} pts")
print(f"  full delta vs source-weighted (failed):        {(full_score-SOURCE_WEIGHTED_SCORES['full'])*100:+.2f} pts")
print(f"  hf_lung-only delta vs unweighted combined:     {(hf_score-UNWEIGHTED_COMBINED_SCORES['hf_lung'])*100:+.2f} pts")
print(f"  hf_lung-only delta vs source-weighted (failed):{(hf_score-SOURCE_WEIGHTED_SCORES['hf_lung'])*100:+.2f} pts")
print()
print(f'  Baseline Normal->Crackle (val, icbhi-only, sem-6): {BASELINE_NORMAL_TO_CRACKLE[0]}/{BASELINE_NORMAL_TO_CRACKLE[1]}')
print(f'  This run — FULL test Normal->Crackle:               {full_cm[0,1]}/{full_cm[0].sum()}')
print(f'  This run — icbhi-only test Normal->Crackle:         {icbhi_cm[0,1]}/{icbhi_cm[0].sum()}')
print(f'  This run — hf_lung-only test Normal->Crackle:       {hf_cm[0,1]}/{hf_cm[0].sum()}')

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, (name, cm) in zip(axes, [('Full test', full_cm), ('ICBHI-only test', icbhi_cm),
                                  ('HF_Lung-only test', hf_cm)]):
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, xticklabels=SOUND_NAMES,
                yticklabels=SOUND_NAMES, cmap='Blues')
    ax.set_title(name); ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_test_confusion_matrices.png', dpi=150)
print(f'\nSaved test confusion matrices -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_test_confusion_matrices.png')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax = axes[0]
ax.plot(history.history['sound_accuracy'], label='Train Acc')
ax.plot(history.history['val_sound_accuracy'], label='Val Acc')
ax.set_title('Sound Accuracy'); ax.set_xlabel('Epoch'); ax.legend()

ax = axes[1]
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_title('Total Loss'); ax.set_xlabel('Epoch'); ax.legend()

ax = axes[2]
ax.plot(history.history['val_icbhi'], label='Val ICBHI', color='green')
ax.axhline(y=max(history.history['val_icbhi']), color='red', linestyle='--',
           label=f'Best: {max(history.history["val_icbhi"]):.4f}')
ax.set_title('ICBHI Score'); ax.set_xlabel('Epoch'); ax.legend()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_training_curves.png', dpi=150)
print(f'Saved training curves -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_training_curves.png')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cm_sound = confusion_matrix(y_sv, y_pred, labels=list(range(NUM_SOUND)))
sns.heatmap(cm_sound, annot=True, fmt='d', ax=axes[0],
            xticklabels=SOUND_NAMES, yticklabels=SOUND_NAMES, cmap='Blues')
axes[0].set_title('Sound Confusion Matrix')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

cm_diag = confusion_matrix(y_dv[diag_mask_val], y_pred_d[diag_mask_val], labels=list(range(NUM_DIAGNOSIS)))
sns.heatmap(cm_diag, annot=True, fmt='d', ax=axes[1],
            xticklabels=DIAG_NAMES, yticklabels=DIAG_NAMES, cmap='Oranges')
axes[1].set_title('Diagnosis Confusion Matrix')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multitask_{RUN_SUFFIX}_confusion_matrices.png', dpi=150)
print(f'Saved confusion matrices -> {RESULTS_DIR}/multitask_{RUN_SUFFIX}_confusion_matrices.png')

Overwriting scripts/train_multitask.py


In [ ]:
!python scripts/train_multitask.py

Train: 27810 | Val: 9106
Train distribution:
sound_label
0    12216
1     9977
2     5197
3      420
Name: count, dtype: int64
Loading val...
Val: (9106, 128, 63, 1)
Loading train...
Train: (27810, 128, 63, 1)
Diagnosis-labeled rows: train 2891/27810, val 799/9106
2026-08-31 20:56:57.007855: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1788209817.009312   20128 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Class weights: [0.5691306483300589, 0.6968527613511075, 1.3377910332884357, 16.553571428571427]
Sound sample_weight (class-weight only, no source multiplier): min=0.569, max=16.554, mean=1.000
Stratified batches: 10 icbhi + 22 hf_lung per 32-sample batch (realized fraction 0.3125, target 0.3) -- p